Import components

In [67]:
import polars as pl
import os
import datetime
import deltalake
import matplotlib.pyplot as plt
import seaborn as sns
import mysql.connector

plt.style.use("ggplot")
sns.set_theme()

db = mysql.connector.connect(
    host=os.getenv("MARIADB_HOST"),
    port=int(os.getenv("MARIADB_PORT", "3306")),
    user=os.getenv("MARIADB_USER"),
    password=os.getenv("MARIADB_PASSWORD"),
    database=os.getenv("MARIADB_DATABASE")
)

<h1 style="font-size: 40px;">Define what the "today" date to inspect is</h>

In [68]:
target_date = datetime.date(2026, 3, 7)
print("Using business_date =", target_date)


Using business_date = 2026-03-07


<h1 style="font-size: 40px;">Load Silver Sales Delta Lake table from MinIO Lakehouse bucket</h>

In [ ]:
table = deltalake.DeltaTable(
    "s3://lakehouse/silver/sales_items",
    storage_options={
        "AWS_ENDPOINT_URL": "http://[IP of MinIO]:9000",
        "AWS_REGION": "local",
        "AWS_S3_FORCE_PATH_STYLE": "true",
        "AWS_ALLOW_HTTP": "true",
    },
)


<h1 style="font-size: 40px;">Filter the Delta Lake table partition to reveal data from just that business date's "today"</h>

In [70]:
arrow_table = table.to_pyarrow_table(
    filters=[("business_date", "=", target_date)]
)

df_today = pl.from_arrow(arrow_table)
df_today


store_id,customer_id,business_date,event_ts,isbn,title,genre,quantity,unit_price,line_total
str,str,date,"datetime[μs, UTC]",str,str,str,i32,f64,f64
"""madison""","""7f65b37c-73d7-46f8-8bf2-536ad0…",2026-03-07,2026-03-07 11:29:22.645334 UTC,"""9780374533557""","""Thinking, Fast And Slow""","""Non-Fiction""",1,29.28,29.28
"""madison""","""7f65b37c-73d7-46f8-8bf2-536ad0…",2026-03-07,2026-03-07 11:29:22.645334 UTC,"""9780593593806""","""Fourth Wing""","""Fantasy""",1,24.99,24.99
"""madison""","""7f65b37c-73d7-46f8-8bf2-536ad0…",2026-03-07,2026-03-07 11:29:22.645334 UTC,"""9780553386790""","""Dune""","""Sci-Fi""",1,18.99,18.99
"""madison""","""beae54cb-3571-4116-b171-a5c71b…",2026-03-07,2026-03-07 15:22:59.312429 UTC,null,null,null,null,null,null
"""madison""","""02fbc02d-284f-40a3-b7eb-91ed3d…",2026-03-07,2026-03-07 10:02:07.327925 UTC,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…
"""madison""","""47b110d3-9e28-4fe6-ba94-e08145…",2026-03-07,2026-03-07 12:03:02.144158 UTC,"""9780553386790""","""Dune""","""Sci-Fi""",1,18.99,18.99
"""glendale""","""3aebb660-2d03-4589-abc3-44cb6e…",2026-03-07,2026-03-07 11:43:40.714266 UTC,"""9781640635333""","""The Last Letter""","""Fiction""",1,27.78,27.78
"""glendale""","""3aebb660-2d03-4589-abc3-44cb6e…",2026-03-07,2026-03-07 11:43:40.714266 UTC,"""9781335534637""","""Heated Rivalry""","""Fiction""",1,31.23,31.23


<h1 style="font-size: 40px;">Inspect the table schema</h>

In [71]:
df_today.schema

Schema([('store_id', String),
        ('customer_id', String),
        ('business_date', Date),
        ('event_ts', Datetime(time_unit='us', time_zone='UTC')),
        ('isbn', String),
        ('title', String),
        ('genre', String),
        ('quantity', Int32),
        ('unit_price', Float64),
        ('line_total', Float64)])

<h1 style="font-size: 40px;">1. TOTAL SALES TODAY</h>

In [72]:
total_sales_today = (
    df_today.select(pl.col("line_total").sum().alias("total_sales_today"))
)

total_sales_today
value = total_sales_today["total_sales_today"][0]
print(f"Total sales for {target_date}: ${value:,.2f}")

Total sales for 2026-03-07: $12,466.58


<h1 style="font-size: 40px;">2. SALES ACROSS 3 STORES PRIOR 6 DAYS + TODAY</h>

In [73]:
full_arrow = table.to_pyarrow_table()
df_all = pl.from_arrow(full_arrow)
start_date = target_date - datetime.timedelta(days=7)

print(f"7-day window: {start_date} → {target_date}")

df_last_7 = df_all.filter(
    (pl.col("business_date") >= start_date) &
    (pl.col("business_date") <= target_date)
)

sales_last_7 = (
    df_last_7
    .group_by(["store_id", "business_date"])
    .agg(pl.col("line_total").sum().alias("total_sales"))
    .sort(["business_date", "store_id"])
)

sales_last_7.to_pandas()

7-day window: 2026-02-28 → 2026-03-07


,store_id,business_date,total_sales
0,brookfield,2026-03-01,4076.53
1,glendale,2026-03-01,4194.16
2,madison,2026-03-01,3542.06
3,brookfield,2026-03-02,5087.91
4,glendale,2026-03-02,4072.31
5,madison,2026-03-02,4583.56
6,brookfield,2026-03-03,3947.04
7,glendale,2026-03-03,3890.46
8,madison,2026-03-03,4230.02
9,brookfield,2026-03-04,4290.45


<h1 style="font-size: 40px;">3. TOP 3 BOOKS SOLD TODAY</h>

In [74]:
arrow_today = table.to_pyarrow_table(
    filters=[("business_date", "=", target_date)]
)

df_today = pl.from_arrow(arrow_today)

top_books_today = (
    df_today
    .group_by(["isbn", "title"])
    .agg([
        pl.col("quantity").sum().alias("total_quantity"),
        pl.col("line_total").sum().alias("total_revenue")
    ])
    .sort("total_quantity", descending=True)
    .head(3)
)

top_books_today.to_pandas()

,isbn,title,total_quantity,total_revenue
0,9781426222686,National Geographic Destinations,54,1890.00
1,9780765376671,The Three-Body Problem,36,647.64
2,9780500545567,Humans of New York,29,868.55


<h1 style="font-size: 40px;">4. BIGGEST SINGLE INCREASE AND BIGGEST SINGLE DECREASE IN BOOK POPULARITY OVER THE LAST 7 DAYS</h>

In [75]:
query = """
    SELECT 
        isbn,
        title,
        base_popularity,
        real_world_popularity,
        sales_popularity,
        snapshot_date
    FROM books_history
"""

df_hist = pl.read_database(query, connection=db)

df_hist = df_hist.with_columns([
    (pl.col("base_popularity") 
     + pl.col("real_world_popularity") 
     + pl.col("sales_popularity")).alias("combined_popularity")
])

df_hist = df_hist.sort(["isbn", "snapshot_date"])

df_hist = df_hist.with_columns([
    pl.col("combined_popularity")
      .shift(1)
      .over("isbn")
      .alias("prev_popularity")
])

df_hist = df_hist.with_columns([
    (pl.col("combined_popularity") - pl.col("prev_popularity"))
      .alias("popularity_delta")
])


max_day = df_hist["snapshot_date"].max()
seven_days_ago = max_day - datetime.timedelta(days=7)

df_7 = df_hist.filter(pl.col("snapshot_date") >= seven_days_ago)

biggest_increase = (
    df_7
    .filter(pl.col("prev_popularity").is_not_null())
    .sort("popularity_delta", descending=True)
    .head(1)
)

biggest_decrease = (
    df_7
    .filter(pl.col("prev_popularity").is_not_null())
    .sort("popularity_delta", descending=False)
    .head(1)
)

print("Biggest 7-Day Increase in Popularity")
display(biggest_increase.to_pandas())

print("Biggest 7-Day Decrease in Popularity")
display(biggest_decrease.to_pandas())

Biggest 7-Day Increase in Popularity


,isbn,title,base_popularity,real_world_popularity,sales_popularity,snapshot_date,combined_popularity,prev_popularity,popularity_delta
0,9781426222686,National Geographic Destinations,11.1308,0.0,14.76,2026-03-02,25.8908,18.3868,7.504


Biggest 7-Day Decrease in Popularity


,isbn,title,base_popularity,real_world_popularity,sales_popularity,snapshot_date,combined_popularity,prev_popularity,popularity_delta
0,9780143127741,The Body Keeps The Score,38.0347,71.3458,4.19296,2026-03-05,113.57346,118.5442,-4.97074


<h1 style="font-size: 40px;">5. TOP 5 BEST SELLING AUTHORS OVER THE LAST 7 DAYS</h>

In [76]:
books_df = pl.read_database(
    "SELECT isbn, author FROM books",
    connection=db
)

sales_with_authors = (
    df_last_7
    .join(books_df, on="isbn", how="left")
)

top_authors_7d = (
    sales_with_authors
    .group_by("author")
    .agg([
        pl.col("quantity").sum().alias("total_quantity"),
        pl.col("line_total").sum().alias("total_revenue")
    ])
    .sort("total_quantity", descending=True)
    .head(5)
)

top_authors_7d.to_pandas()



,author,total_quantity,total_revenue
0,National Geographic,278,9730.00
1,Rebecca Yarros,253,6777.11
2,Brandon Stanton,242,7247.90
3,Liu Cixin,217,3903.83
4,Eric Carle,188,1690.12


<h1 style="font-size: 40px;">FINAL: EXPORT ALL RESULTS TO A SINGLE EXCEL WORKBOOK</h>

In [ ]:
import pandas as pd

date_str = target_date.strftime("%Y-%m-%d")
output_path = f"data/final/bookstore_metrics_{date_str}.xlsx"

with pd.ExcelWriter(output_path, engine="xlsxwriter") as writer:
    total_sales_today.to_pandas().to_excel(writer, sheet_name="Total Sales Today", index=False)
    sales_last_7.to_pandas().to_excel(writer, sheet_name="Sales Over Time", index=False)
    top_books_today.to_pandas().to_excel(writer, sheet_name="Top 3 Books Today", index=False)
    biggest_increase.to_pandas().to_excel(writer, sheet_name="Biggest Increase", index=False)
    biggest_decrease.to_pandas().to_excel(writer, sheet_name="Biggest Decrease", index=False)
    top_authors_7d.to_pandas().to_excel(writer, sheet_name="Top Authors 7 Days", index=False)